In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"


spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}"
)


print(
    f"Bronze source: {catalog}.{bronze_schema}"
)

print(
    f"Silver destination: {catalog}.{silver_schema}"
)


display(
    spark.sql(
        f"SHOW TABLES IN {catalog}.{bronze_schema}"
    )
)

#### Binance trades

In [ ]:
spot_bronze_df = spark.table(f"{catalog}.{bronze_schema}.binance_spot_trades")
convert_bronze_df = spark.table(f"{catalog}.{bronze_schema}.binance_convert_trades")

print("Spot rows",spot_bronze_df.count())
print("Convert rows",convert_bronze_df.count())

display(spot_bronze_df.limit(5))
display(convert_bronze_df.limit(5))

In [ ]:
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)


decimal_type = DecimalType(38, 18)


spot_clean_df = (
    spot_bronze_df
    .select(
        col("id").cast("string").alias("transaction_id"),
        col("orderId").cast("string").alias("order_id"),
        col("symbol"),
        when(
            col("isBuyer"),
            lit("BUY")
        ).otherwise(
            lit("SELL")
        ).alias("side"),
        col("price").cast(decimal_type).alias("price"),
        col("qty").cast(decimal_type).alias("quantity"),
        col("quoteQty").cast(decimal_type).alias("quote_amount"),
        col("commission").cast(decimal_type).alias("fee_amount"),
        col("commissionAsset").alias("fee_asset"),
        col("isMaker").alias("is_maker"),
        timestamp_millis(
            col("time")
        ).alias("trade_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)


spot_clean_df.printSchema()

display(
    spot_clean_df.orderBy(
        col("trade_time").desc()
    )
)

In [ ]:
# Normalize Spot purchases and sales

spot_normalized_df = (
    spot_clean_df
    .withColumn(
        "base_asset",
        regexp_replace(
            col("symbol"),
            "USDT$",
            ""
        )
    )
    .withColumn(
        "quote_asset",
        lit("USDT")
    )
    .withColumn(
        "asset_spent",
        when(
            col("side") == "BUY",
            col("quote_asset")
        ).otherwise(
            col("base_asset")
        )
    )
    .withColumn(
        "amount_spent",
        when(
            col("side") == "BUY",
            col("quote_amount")
        ).otherwise(
            col("quantity")
        )
    )
    .withColumn(
        "asset_received",
        when(
            col("side") == "BUY",
            col("base_asset")
        ).otherwise(
            col("quote_asset")
        )
    )
    .withColumn(
        "amount_received",
        when(
            col("side") == "BUY",
            col("quantity")
        ).otherwise(
            col("quote_amount")
        )
    )
    .withColumn(
        "trade_source",
        lit("Spot")
    )
)


display(
    spot_normalized_df.select(
        "transaction_id",
        "side",
        "asset_spent",
        "amount_spent",
        "asset_received",
        "amount_received",
        "price",
        "fee_amount",
        "fee_asset",
        "trade_time"
    )
)

In [ ]:
convert_clean_df = (
    convert_bronze_df
    .select(
        col("orderId")
            .cast("string")
            .alias("transaction_id"),

        lit("Convert")
            .alias("trade_source"),

        lit(None)
            .cast("string")
            .alias("symbol"),

        col("side"),

        col("fromAsset")
            .alias("asset_spent"),

        col("fromAmount")
            .cast(decimal_type)
            .alias("amount_spent"),

        col("toAsset")
            .alias("asset_received"),

        col("toAmount")
            .cast(decimal_type)
            .alias("amount_received"),

        lit(None)
            .cast(decimal_type)
            .alias("fee_amount"),

        lit(None)
            .cast("string")
            .alias("fee_asset"),

        timestamp_millis(
            col("createTime")
        ).alias("trade_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
            .alias("bronze_ingested_at")
    )
)

display(convert_clean_df)

In [ ]:
spot_for_trades_df = (
    spot_normalized_df
    .select(
        col("transaction_id"),
        col("trade_source"),
        col("symbol"),
        col("side"),

        col("asset_spent"),
        col("amount_spent"),

        col("asset_received"),
        col("amount_received"),

        col("fee_amount"),
        col("fee_asset"),

        col("trade_time"),
        col("exchange"),
        col("_source_file"),
        col("bronze_ingested_at")
    )
)

display(spot_for_trades_df)

In [ ]:
silver_trades_df = (
    spot_for_trades_df
    .unionByName(
        convert_clean_df
    )
    .withColumn(
        "trade_key",
        concat_ws(
            "|",
            col("exchange"),
            col("trade_source"),
            coalesce(
                col("symbol"),
                lit("NO_SYMBOL")
            ),
            col("transaction_id")
        )
    )
)

print("Total Silver trades:", silver_trades_df.count())

display(
    silver_trades_df
    .select(
        "trade_key",
        "transaction_id",
        "trade_source",
        "symbol",
        "side",
        "asset_spent",
        "amount_spent",
        "asset_received",
        "amount_received",
        "trade_time"
    )
    .orderBy(
        col("trade_time").desc()
    )
)

In [ ]:
trades_before = silver_trades_df.count()

silver_trades_clean_df = (
    silver_trades_df
    .filter(
        col("transaction_id").isNotNull()
        & col("asset_spent").isNotNull()
        & col("asset_received").isNotNull()
        & (col("amount_spent") > 0)
        & (col("amount_received") > 0)
    )
    .dropDuplicates([
        "trade_key"
    ])
    .withColumn(
        "trade_date",
        to_date(col("trade_time"))
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)


trades_after = silver_trades_clean_df.count()


print("Rows before cleaning:", trades_before)
print("Rows after cleaning:", trades_after)
print("Rows removed:", trades_before - trades_after)

In [ ]:
(
    silver_trades_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_trades"
    )
)

In [ ]:
display(
    spark.table(
        f"{catalog}.{silver_schema}.binance_trades"
    )
)

####binance p2p

In [ ]:
p2p_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_p2p_trades"
)


print(
    "P2P Bronze rows:",
    p2p_bronze_df.count()
)


display(
    p2p_bronze_df
)

In [ ]:
p2p_silver_df = (
    p2p_bronze_df
    .select(
        col("orderNumber")
        .alias("p2p_order_id"),

        col("tradeType")
        .alias("trade_type"),

        col("asset"),

        col("fiat"),

        col("amount")
        .cast(decimal_type)
        .alias("asset_amount"),

        col("totalPrice")
        .cast(decimal_type)
        .alias("fiat_amount"),

        col("unitPrice")
        .cast(decimal_type)
        .alias("unit_price"),

        col("takerCommission")
        .cast(decimal_type)
        .alias("fee_amount"),

        col("takerAmount")
        .cast(decimal_type)
        .alias("net_asset_amount"),

        col("orderStatus")
        .alias("status"),

        timestamp_millis(
            col("createTime")
        ).alias("transaction_time"),

        col("payMethodName")
        .alias("payment_method"),

        col("exchange"),

        col("_source_file"),

        col("ingested_at").alias(
            "bronze_ingested_at"
        )
    )
    .dropDuplicates([
        "p2p_order_id"
    ])
    .withColumn(
        "is_completed",
        col("status") == "COMPLETED"
    )
    .withColumn(
        "transaction_date",
        to_date(col("transaction_time"))
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)

print(
    "P2P Silver rows:",
    p2p_silver_df.count()
)

display(
    p2p_silver_df.orderBy(
        col("transaction_time").desc()
    )
)

In [ ]:
(
    p2p_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_p2p_transactions"
    )
)

display(
    spark.table(
        f"{catalog}.{silver_schema}.binance_p2p_transactions"
    )
)

####Binance account

In [ ]:
account_bronze_df  = spark.table(
    f"{catalog}.{bronze_schema}.binance_account_snapshots"
)

print(
    "Bronze account rows",account_bronze_df.count()
)

account_bronze_df.printSchema()

display(account_bronze_df)

In [ ]:
account_snapshots_silver_df = (
    account_bronze_df
    .select(
        col("source_file_name").alias("snapshot_id"),
        col("uid").cast("string").alias("account_id"),
        col("accountType").alias("account_type"),

        col("canTrade").alias("can_trade"),
        col("canWithdraw").alias("can_withdraw"),
        col("canDeposit").alias("can_deposit"),

        col("commissionRates.maker")
            .cast("decimal(18,8)")
            .alias("maker_commission_rate"),

        col("commissionRates.taker")
            .cast("decimal(18,8)")
            .alias("taker_commission_rate"),

        col("commissionRates.buyer")
            .cast("decimal(18,8)")
            .alias("buyer_commission_rate"),

        col("commissionRates.seller")
            .cast("decimal(18,8)")
            .alias("seller_commission_rate"),

        when(
            col("updateTime") > 0,
            timestamp_millis(
                col("updateTime")
            )
        ).alias("account_updated_at"),

        col("permissions"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)

print(
    "Clean account snapshot rows:",
    account_snapshots_silver_df.count()
)

display(account_snapshots_silver_df)

In [ ]:
rows_before = account_snapshots_silver_df.count()

account_snapshots_final_df = (
    account_snapshots_silver_df
    .dropDuplicates([
        "snapshot_id"
    ])
)

rows_after = account_snapshots_final_df.count()

print("Rows before deduplication:", rows_before)
print("Rows after deduplication:", rows_after)
print("Duplicate rows removed:", rows_before - rows_after)

In [ ]:
(
    account_snapshots_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_account_snapshots"
    )
)
print(
    "Saved account snapshots:",
    spark.table(
        f"{catalog}.{silver_schema}.binance_account_snapshots"
    ).count()
)

####Binance balances

In [ ]:
account_balances_silver_df = (
    account_bronze_df
    .select(
        col("source_file_name").alias("snapshot_id"),
        col("uid").cast("string").alias("account_id"),
        explode(col("balances")).alias("balance"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .select(
        "snapshot_id",
        "account_id",

        upper(col("balance.asset")).alias("asset"),

        col("balance.free")
            .cast("decimal(38,18)")
            .alias("free_balance"),

        col("balance.locked")
            .cast("decimal(38,18)")
            .alias("locked_balance"),

        "exchange",
        "_source_file",
        "bronze_ingested_at"
    )
    .withColumn(
        "total_balance",
        col("free_balance") + col("locked_balance")
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)

print(
    "Account balance rows:",
    account_balances_silver_df.count()
)

display(
    account_balances_silver_df.orderBy(
        col("total_balance").desc()
    )
)

In [ ]:
active_balances_df = (
    account_balances_silver_df
    .filter(col("total_balance") > 0)
)

total_rows = account_balances_silver_df.count()
active_rows = active_balances_df.count()
zero_rows = total_rows - active_rows

print("Total balance rows:", total_rows)
print("Assets with funds:", active_rows)
print("Assets with zero balance:", zero_rows)

display(
    active_balances_df.orderBy(
        col("total_balance").desc()
    )
)

In [ ]:
account_balances_final_df = (
    account_balances_silver_df
    .dropDuplicates(
        ["snapshot_id", "account_id", "asset"]
    )
)

(
    account_balances_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_account_balances"
    )
)

saved_balance_rows = spark.table(
    f"{catalog}.{silver_schema}.binance_account_balances"
).count()

print("Saved account balance rows:", saved_balance_rows)

####Binance deposits

In [ ]:
deposits_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_deposits"
)

print(
    "Deposits rows: ",
    deposits_bronze_df.count()
)

deposits_bronze_df.printSchema()
display(deposits_bronze_df)

In [ ]:
deposits_silver_df = (
    deposits_bronze_df
    .select(
        col("id").alias("deposit_id"),
        col("txId").alias("transaction_hash"),
        upper(col("coin")).alias("asset"),
        col("amount")
            .cast("decimal(38,18)")
            .alias("amount"),
        upper(col("network")).alias("network"),
        col("status")
            .cast("integer")
            .alias("status_code"),

        col("address").alias("deposit_address"),
        col("addressTag").alias("address_tag"),
        col("sourceAddress").alias("source_address"),

        when(
            col("insertTime") > 0,
            timestamp_millis(
                col("insertTime")
            )
        ).alias("deposit_time"),

        when(
            col("completeTime") > 0,
            timestamp_millis(
                col("completeTime")
            )
        ).alias("completed_at"),

        col("transferType")
            .cast("integer")
            .alias("transfer_type"),

        col("confirmTimes").alias("confirmations"),

        col("unlockConfirm")
            .cast("integer")
            .alias("unlock_confirmations"),

        col("walletType")
            .cast("integer")
            .alias("wallet_type"),

        col("travelRuleStatus")
            .cast("integer")
            .alias("travel_rule_status"),

        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)

print("Clean deposit rows:", deposits_silver_df.count())

deposits_silver_df.printSchema()

In [ ]:
deposits_final_df = (
    deposits_silver_df
    .dropDuplicates(["deposit_id"])
)

print(
    "Duplicate deposit rows removed:",
    deposits_silver_df.count() - deposits_final_df.count()
)

(
    deposits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_deposits"
    )
)

saved_deposit_rows = spark.table(
    f"{catalog}.{silver_schema}.binance_deposits"
).count()

print("Saved deposit rows:", saved_deposit_rows)

####Binance withdrawals

In [ ]:
withdrawals_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_withdrawals"
)

print(
    "Bronze withdrawal rows:",
    withdrawals_bronze_df.count()
)

withdrawals_bronze_df.printSchema()

display(withdrawals_bronze_df)

In [ ]:
withdrawals_silver_df = (
    withdrawals_bronze_df
    .select(
        col("id").alias("withdrawal_id"),
        col("withdrawOrderId").alias("withdrawal_order_id"),
        col("txId").alias("transaction_hash"),

        upper(col("coin")).alias("asset"),

        col("amount")
            .cast("decimal(38,18)")
            .alias("amount"),

        col("transactionFee")
            .cast("decimal(38,18)")
            .alias("transaction_fee"),

        upper(col("network")).alias("network"),

        col("status")
            .cast("integer")
            .alias("status_code"),

        col("address").alias("withdrawal_address"),

        to_timestamp(
            col("applyTime")
        ).alias("requested_at"),

        to_timestamp(
            col("completeTime")
        ).alias("completed_at"),

        col("confirmNo")
            .cast("integer")
            .alias("confirmation_count"),

        col("transferType")
            .cast("integer")
            .alias("transfer_type"),

        col("walletType")
            .cast("integer")
            .alias("wallet_type"),

        col("txKey").alias("transaction_key"),
        col("info").alias("status_information"),

        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)

print(
    "Clean withdrawal rows:",
    withdrawals_silver_df.count()
)

display(withdrawals_silver_df)

In [ ]:
withdrawal_rows_before = withdrawals_silver_df.count()

withdrawals_final_df = (
    withdrawals_silver_df
    .dropDuplicates(["withdrawal_id"])
)

withdrawal_rows_after = withdrawals_final_df.count()

print("Rows before deduplication:", withdrawal_rows_before)
print("Rows after deduplication:", withdrawal_rows_after)
print(
    "Duplicate withdrawal rows removed:",
    withdrawal_rows_before - withdrawal_rows_after
)

In [ ]:
(
    withdrawals_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_withdrawals"
    )
)

saved_withdrawal_rows = spark.table(
    f"{catalog}.{silver_schema}.binance_withdrawals"
).count()

print("Saved withdrawal rows:", saved_withdrawal_rows)

####Binance internal transfer

In [ ]:
transfers_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_transfers"
)

print(
    "Bronze transfer rows:",
    transfers_bronze_df.count()
)

transfers_bronze_df.printSchema()

display(transfers_bronze_df)

In [ ]:
transfers_silver_df = (
    transfers_bronze_df
    .select(
        col("tranId")
            .cast("string")
            .alias("transfer_id"),

        upper(col("asset")).alias("asset"),

        col("amount")
            .cast("decimal(38,18)")
            .alias("amount"),

        upper(col("type")).alias("transfer_type"),

        upper(col("transferType"))
            .alias("requested_transfer_type"),

        upper(col("status")).alias("status"),

        when(
            col("timestamp") > 0,
            timestamp_millis(
                col("timestamp")
            )
        ).alias("transfer_time"),

        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at"),
        current_timestamp().alias("silver_processed_at")
    )
)

print(
    "Clean transfer rows:",
    transfers_silver_df.count()
)

transfers_silver_df.printSchema()

In [ ]:
transfers_final_df = (
    transfers_silver_df
    .dropDuplicates(["transfer_id"])
)

print(
    "Duplicate transfer rows removed:",
    transfers_silver_df.count() - transfers_final_df.count()
)

(
    transfers_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_transfers"
    )
)

saved_transfer_rows = spark.table(
    f"{catalog}.{silver_schema}.binance_transfers"
).count()

print("Saved transfer rows:", saved_transfer_rows)

####Binance price history

In [ ]:
price_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_price_history"
)

print("Bronze price rows:", price_bronze_df.count())

print("BRONZE SCHEMA")
price_bronze_df.printSchema()

In [ ]:
price_silver_df = (
    price_bronze_df
    .select(
        upper(
            col("symbol")
        ).alias("symbol"),

        regexp_replace(
            upper(col("symbol")),
            "USDT$",
            ""
        ).alias("asset"),

        timestamp_millis(
            col("open_time").cast("long")
        ).alias("open_time"),

        timestamp_millis(
            col("close_time").cast("long")
        ).alias("close_time"),

        col("open_price")
            .cast("decimal(38,18)")
            .alias("open_price"),

        col("high_price")
            .cast("decimal(38,18)")
            .alias("high_price"),

        col("low_price")
            .cast("decimal(38,18)")
            .alias("low_price"),

        col("close_price")
            .cast("decimal(38,18)")
            .alias("close_price"),

        col("volume")
            .cast("decimal(38,18)")
            .alias("volume"),

        col("quote_asset_volume")
            .cast("decimal(38,18)")
            .alias("quote_asset_volume"),

        col("number_of_trades")
            .cast("long")
            .alias("trade_count"),

        col("taker_buy_base_volume")
            .cast("decimal(38,18)")
            .alias("taker_buy_base_volume"),

        col("taker_buy_quote_volume")
            .cast("decimal(38,18)")
            .alias("taker_buy_quote_volume"),

        col("exchange"),
        col("_source_file"),
        col("source_file_modified_at"),
        col("source_file_saved_at"),
        col("ingested_at")
            .alias("bronze_ingested_at"),

        current_timestamp()
            .alias("silver_processed_at")
    )
)

print(
    "Clean price rows:",
    price_silver_df.count()
)

display(
    price_silver_df.orderBy(
        col("open_time").desc()
    )
)

In [ ]:
price_enriched_df = (
    price_silver_df
    .withColumn(
        "is_closed",
        col("close_time") <= col("source_file_saved_at")
    )
    .withColumn(
        "price_change",
        (
            col("close_price") - col("open_price")
        ).cast("decimal(38,18)")
    )
    .withColumn(
        "price_change_pct",
        when(
            col("open_price") != 0,
            (
                (
                    col("close_price") - col("open_price")
                ) / col("open_price")
            ) * 100
        )
        .otherwise(None)
        .cast("decimal(38,18)")
    )
)


price_rows_before = price_enriched_df.count()


price_final_df = (
    price_enriched_df
    .dropDuplicates([
        "symbol",
        "open_time"
    ])
)


price_rows_after = price_final_df.count()


print("Rows before deduplication:", price_rows_before)
print("Rows after deduplication:", price_rows_after)
print(
    "Duplicate candles removed:",
    price_rows_before - price_rows_after
)


price_final_df.groupBy(
    "is_closed"
).count().show()

In [ ]:
(
    price_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_price_history"
    )
)

print(
    "Saved price history rows:",
    spark.table(
        f"{catalog}.{silver_schema}.binance_price_history"
    ).count()
)

####Binance funding account snapshots

In [ ]:
funding_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.binance_funding_account_snapshots"
)

funding_snapshots_silver_df = (
    funding_bronze_df
    .select(
        col("source_file_name").alias(
            "snapshot_id"
        ),

        upper(col("walletType")).alias(
            "wallet_type"
        ),

        to_timestamp(
            col("capturedAt")
        ).alias("snapshot_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at").alias(
            "bronze_ingested_at"
        ),

        current_timestamp().alias(
            "silver_processed_at"
        )
    )
    .dropDuplicates([
        "snapshot_id"
    ])
)

print(
    "Funding snapshot rows:",
    funding_snapshots_silver_df.count()
)

display(
    funding_snapshots_silver_df.orderBy(
        col("snapshot_time").desc()
    )
)

In [ ]:
(
    funding_snapshots_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_funding_account_snapshots"
    )
)

saved_funding_snapshots_df = spark.table(
    f"{catalog}.{silver_schema}.binance_funding_account_snapshots"
)

print(
    "Saved Funding snapshot rows:",
    saved_funding_snapshots_df.count()
)

#### Binance funding account balances

In [ ]:
funding_balances_silver_df = (
    funding_bronze_df
    .select(
        col("source_file_name").alias(
            "snapshot_id"
        ),

        upper(col("walletType")).alias(
            "wallet_type"
        ),

        to_timestamp(
            col("capturedAt")
        ).alias("snapshot_time"),

        explode(col("balances")).alias(
            "balance"
        ),

        col("exchange"),
        col("_source_file"),

        col("ingested_at").alias(
            "bronze_ingested_at"
        )
    )
    .select(
        "snapshot_id",
        "wallet_type",
        "snapshot_time",

        upper(col("balance.asset")).alias(
            "asset"
        ),

        col("balance.free")
        .cast("decimal(38,18)")
        .alias("free_balance"),

        col("balance.locked")
        .cast("decimal(38,18)")
        .alias("locked_balance"),

        col("balance.freeze")
        .cast("decimal(38,18)")
        .alias("frozen_balance"),

        col("balance.withdrawing")
        .cast("decimal(38,18)")
        .alias("withdrawing_balance"),

        col("exchange"),
        col("_source_file"),
        col("bronze_ingested_at")
    )
    .withColumn(
        "total_balance",

        coalesce(
            col("free_balance"),
            lit(0)
        )
        + coalesce(
            col("locked_balance"),
            lit(0)
        )
        + coalesce(
            col("frozen_balance"),
            lit(0)
        )
        + coalesce(
            col("withdrawing_balance"),
            lit(0)
        )
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates([
        "snapshot_id",
        "asset"
    ])
)

print(
    "Funding balance rows:",
    funding_balances_silver_df.count()
)

funding_balances_silver_df.printSchema()

display(funding_balances_silver_df)

In [ ]:
(
    funding_balances_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.binance_funding_account_balances"
    )
)

saved_funding_balances_df = spark.table(
    f"{catalog}.{silver_schema}.binance_funding_account_balances"
)

print(
    "Saved Funding balance rows:",
    saved_funding_balances_df.count()
)

In [ ]:
silver_tables = spark.sql(
    "SHOW TABLES IN crypto_lakehouse.silver"
)

display(silver_tables)